In [11]:
import pandas as pd
from datetime import datetime, timedelta

schedule = pd.read_csv('schedule.csv', parse_dates = ['start time'])

schedule['finish'] = schedule['start time'] + pd.to_timedelta(schedule['setup time'] + schedule['run time'], unit = 'h')

wcs = schedule['work center'].unique()

In [12]:
job = int(input("Enter Job Number: "))
part = int(input("Enter Part Number: "))
material = str(input("Aluminum (Enter alm) or Steel? (enter stl)").lower())
due_date = datetime.strptime(input("Enter Due Date (yyyy-mm-dd): "), "%Y-%m-%d")
ops = int(input("How many operations does this job have? "))

rows = []

for x in range(ops):
    y = x + 1
    dept = int(input(f"Which department runs op {y}? "))
    setup_time = float(input("Enter setup time: "))
    run_time = float(input("Enter run time: "))
    row = {"job": job, "part number": part, "material": material, "due date": due_date, "operation sequence": y, "department": dept, "work center": None, "start time": None, "setup time": setup_time, "run time": run_time}
    rows.append(row)



In [ ]:
import json

#-----Load work center capabilities (department + material) so each operation can be matched to a legal work center.

layout = json.load(open('shop_config.json'))

new_rows = pd.DataFrame(rows, columns = ["job", "part number", "material", "due date", "operation sequence", "department", "work center", "start time", "setup time", "run time"])

#-----wc_free tracks, per work center, the latest time it is committed until. Seeded from the existing schedule so new jobs
#-----never get placed on top of an already-scheduled operation. A work center with no history yet is free right now.

wc_free = schedule.groupby('work center')['finish'].max().to_dict()

#-----prev_finish enforces the op-sequence dependency: op N can't start before op N-1 finishes, regardless of which
#-----department/work center each one lands on. None for op 1 since it has no predecessor.

prev_finish = None

for idx, row in new_rows.iterrows():
    #-----Walk every work center in the shop layout and keep only the ones that can legally run this operation:
    #-----same department as the op, and the op's material is one this work center is capable of running.
    candidates = []
    for wc in layout:
        if wc['department'] == row['department'] and row['material'] in wc['capabilities']:
            candidates.append(wc['number'])

    if not candidates:
        raise ValueError(f"No work center in department {row['department']} can run material '{row['material']}' (op {row['operation sequence']})")

    best_wc = best_start = best_finish = None

    for wc in candidates:
        wc_available = wc_free.get(wc, datetime.now())
        start = max(wc_available, prev_finish) if prev_finish is not None else wc_available
        finish = start + timedelta(hours = row['setup time'] + row['run time'])

        if best_finish is None or finish < best_finish:
            best_wc, best_start, best_finish = wc, start, finish

    new_rows.loc[idx, 'work center'] = best_wc
    new_rows.loc[idx, 'start time'] = best_start

    #-----Commit this operation's slot immediately so the next operation (same job or, later, another new job) sees it as busy.

    wc_free[best_wc] = best_finish
    prev_finish = best_finish

new_rows['finish'] = new_rows['start time'] + pd.to_timedelta(new_rows['setup time'] + new_rows['run time'], unit = 'h')

In [14]:
#-----Only append once work center/start time are fully resolved above, so schedule.csv never holds an incomplete row.
#-----'finish' is dropped before writing since it's derived data (see README) and isn't a column in schedule.csv.

new_rows.drop(columns = ['finish']).to_csv('schedule.csv', mode = 'a', header = False, index = False)